In [3]:
import pandas as pd
import requests
import time
import random
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import json
import logging
from urllib.parse import quote
import warnings
warnings.filterwarnings('ignore')

class MenuPopularityAnalyzer:
    def __init__(self, search_year=2024, search_quarter=1):
        # .env 파일에서 API 키 로드
        load_dotenv()
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        if not self.client_id or not self.client_secret:
            raise ValueError("네이버 API 키가 설정되지 않았습니다.")
        
        self.search_year = search_year
        self.search_quarter = search_quarter
        self.search_months = self.get_quarter_months(search_quarter)
        
        self.news_api_url = "https://openapi.naver.com/v1/search/news.json"
        self.headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret,
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        self.menu_popularity = {}
        self.detailed_results = []
        
        self.request_delay = 0.1
        self.daily_request_count = 0
        self.max_daily_requests = 20000
        
        print(f"메뉴 인기도 분석기 초기화 완료")
        print(f"분석 기간: {search_year}년 {search_quarter}분기 ({self.search_months[0]}-{self.search_months[-1]}월)")
    
    def get_quarter_months(self, quarter):
        quarter_map = {
            1: [1, 2, 3],
            2: [4, 5, 6], 
            3: [7, 8, 9],
            4: [10, 11, 12]
        }
        return quarter_map.get(quarter, [1, 2, 3])
    
    def load_menu_data(self, csv_file_path):
        try:
            df = pd.read_csv(csv_file_path, encoding='utf-8')
            print(f"CSV 파일 로드 완료: {len(df)}개 행")
            
            menu_items = []
            for _, row in df.iterrows():
                detail_menus = [menu.strip() for menu in str(row['상세메뉴']).split(',') if menu.strip()]
                for detail_menu in detail_menus:
                    menu_items.append({
                        '대분류': row['대분류'],
                        '중분류': row['중분류'],
                        '소분류': row['소분류'],
                        '상세메뉴': detail_menu,
                        '시각적특징': row['시각적특징']
                    })
            
            print(f"총 {len(menu_items)}개의 개별 메뉴 항목 생성")
            return menu_items
            
        except Exception as e:
            print(f"CSV 파일 로드 실패: {e}")
            raise
    
    def search_news_api(self, query, start=1, display=100):
        try:
            if self.daily_request_count >= self.max_daily_requests:
                print("일일 API 요청 한도에 도달했습니다.")
                return None
            
            params = {
                'query': query,
                'start': start,
                'display': min(display, 100),
                'sort': 'date'
            }
            
            response = requests.get(self.news_api_url, headers=self.headers, params=params, timeout=10)
            self.daily_request_count += 1
            
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                print("API 제한 도달, 10초 대기...")
                time.sleep(10)
                return None
            else:
                print(f"뉴스 API 요청 실패: {response.status_code}")
                return None
                
        except Exception as e:
            print(f"뉴스 API 요청 중 오류: {e}")
            return None
        finally:
            time.sleep(self.request_delay)
    
    def is_valid_news_date(self, pub_date):
        """뉴스 발행일이 지정 분기 내인지 확인"""
        if not pub_date:
            return False
        
        try:
            from email.utils import parsedate_tz
            parsed = parsedate_tz(pub_date.strip())
            if parsed:
                dt = datetime(*parsed[:6])
                return dt.year == self.search_year and dt.month in self.search_months
        except:
            pass
        
        try:
            parts = pub_date.split()
            if len(parts) >= 4:
                year = int(parts[3])
                month_name = parts[2]
                
                month_map = {
                    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
                    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
                    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
                }
                
                month = month_map.get(month_name, 0)
                return year == self.search_year and month in self.search_months
        except:
            pass
        
        return False
    
    def generate_search_keywords(self, menu_name):
        """메뉴별 다양한 검색 키워드 생성"""
        keywords = [
            menu_name,
            f"{menu_name} 맛집",
            f"{menu_name} 요리",
            f"{menu_name} 레시피", 
            f"{menu_name} 추천",
            f"맛있는 {menu_name}",
            f"{menu_name} 음식점",
            f"{menu_name} 전문점"
        ]
        
        return keywords[:6]  # 최대 6개 키워드
    
    def analyze_menu_popularity_comprehensive(self, menu_item):
        """포괄적 메뉴 인기도 분석"""
        menu_name = menu_item['상세메뉴']
        search_keywords = self.generate_search_keywords(menu_name)
        
        total_mentions = 0
        keyword_results = {}
        
        print(f"  '{menu_name}' 분석 중 (키워드 {len(search_keywords)}개)")
        
        for keyword in search_keywords:
            keyword_count = 0
            start = 1
            max_pages = 3  # 키워드당 최대 3페이지
            
            for page in range(max_pages):
                result = self.search_news_api(keyword, start=start, display=100)
                
                if not result or 'items' not in result:
                    break
                
                news_items = result['items']
                if not news_items:
                    break
                
                page_count = 0
                for news in news_items:
                    if self.is_valid_news_date(news.get('pubDate', '')):
                        page_count += 1
                
                keyword_count += page_count
                
                # 해당 분기 뉴스가 없으면 중단
                if page_count == 0:
                    break
                
                start += len(news_items)
                time.sleep(0.1)
            
            keyword_results[keyword] = keyword_count
            total_mentions += keyword_count
            
            if keyword_count > 0:
                print(f"    '{keyword}': {keyword_count}회")
        
        # 상세 결과 저장
        self.detailed_results.append({
            '메뉴명': menu_name,
            '대분류': menu_item['대분류'],
            '중분류': menu_item['중분류'],
            '소분류': menu_item['소분류'],
            '총언급횟수': total_mentions,
            '키워드별결과': keyword_results,
            '분석일시': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        
        return total_mentions
    
    def calculate_fair_quotas(self, menu_items, popularity_scores, total_target=8000):
        """공정한 할당량 계산 (가중치 없음)"""
        total_popularity = sum(popularity_scores.values())
        mentioned_menus = len([score for score in popularity_scores.values() if score > 0])
        
        print(f"\n순수 인기도 분석 결과:")
        print(f"총 언급 횟수: {total_popularity}회")
        print(f"언급된 메뉴: {mentioned_menus}개 / {len(menu_items)}개")
        
        # 기본 할당량 (모든 메뉴에 최소 보장) - 전체의 30%
        base_quota_ratio = 0.3
        base_total = int(total_target * base_quota_ratio)
        base_quota_per_menu = max(1, base_total // len(menu_items))
        
        # 비례 할당량 - 나머지 70%
        proportional_total = total_target - (base_quota_per_menu * len(menu_items))
        
        quotas = {}
        
        if total_popularity > 0:
            for menu_item in menu_items:
                menu_name = menu_item['상세메뉴']
                popularity = popularity_scores.get(menu_name, 0)
                
                # 기본 할당량
                base_quota = base_quota_per_menu
                
                # 인기도 비례 추가 할당
                if popularity > 0:
                    ratio = popularity / total_popularity
                    additional_quota = int(proportional_total * ratio)
                else:
                    additional_quota = 0
                
                total_quota = base_quota + additional_quota
                quotas[menu_name] = total_quota
        else:
            # 인기도 정보가 없으면 균등 배분
            quota_per_menu = total_target // len(menu_items)
            for menu_item in menu_items:
                quotas[menu_item['상세메뉴']] = quota_per_menu
        
        print(f"\n할당량 계산 방식:")
        print(f"기본 할당량: {base_quota_per_menu}개 × {len(menu_items)}메뉴 = {base_quota_per_menu * len(menu_items)}개")
        print(f"비례 할당량: {proportional_total}개 (인기도 비례 배분)")
        print(f"총 할당량: {sum(quotas.values())}개")
        
        return quotas
    
    def analyze_all_menus(self, csv_file_path):
        """모든 메뉴 인기도 분석"""
        menu_items = self.load_menu_data(csv_file_path)
        
        months_str = f"{self.search_months[0]}-{self.search_months[-1]}월"
        print(f"\n{self.search_year}년 {months_str} 메뉴별 순수 인기도 분석 시작")
        print("="*60)
        
        popularity_scores = {}
        
        for i, menu_item in enumerate(menu_items):
            menu_name = menu_item['상세메뉴']
            print(f"\n[{i+1}/{len(menu_items)}] {menu_name}")
            
            mentions = self.analyze_menu_popularity_comprehensive(menu_item)
            popularity_scores[menu_name] = mentions
            
            print(f"  총 언급: {mentions}회")
            
            # 진행 상황 표시
            if (i + 1) % 20 == 0:
                progress = ((i + 1) / len(menu_items)) * 100
                print(f"\n진행률: {progress:.1f}% ({i+1}/{len(menu_items)})")
                print(f"API 요청: {self.daily_request_count}회")
            
            time.sleep(random.uniform(0.3, 0.6))
        
        # 공정한 할당량 계산 (가중치 없음)
        quotas = self.calculate_fair_quotas(menu_items, popularity_scores)
        
        # 결과 저장
        self.menu_popularity = popularity_scores
        
        print(f"\n분석 완료!")
        print(f"API 요청 총 {self.daily_request_count}회 사용")
        
        return popularity_scores, quotas
    
    def save_analysis_results(self, popularity_scores, quotas, output_file="menu_popularity_pure_2024Q1.xlsx"):
        """분석 결과를 엑셀로 저장"""
        try:
            import pandas as pd
            import openpyxl
            from openpyxl.styles import Font, Alignment
            
            # 메뉴별 인기도와 할당량
            results_data = []
            for result in self.detailed_results:
                menu_name = result['메뉴명']
                popularity = popularity_scores.get(menu_name, 0)
                quota = quotas.get(menu_name, 0)
                
                results_data.append({
                    '메뉴명': menu_name,
                    '대분류': result['대분류'],
                    '중분류': result['중분류'],
                    '소분류': result['소분류'],
                    '순수인기도': popularity,
                    '할당량': quota,
                    '할당비율': f"{(quota/sum(quotas.values())*100):.2f}%" if sum(quotas.values()) > 0 else "0%",
                    '분석일시': result['분석일시']
                })
            
            results_df = pd.DataFrame(results_data)
            results_df = results_df.sort_values('순수인기도', ascending=False)
            
            # 인기도 순위 (TOP 50)
            popularity_df = pd.DataFrame(list(popularity_scores.items()), 
                                       columns=['메뉴명', '순수인기도'])
            popularity_df = popularity_df.sort_values('순수인기도', ascending=False)
            popularity_df['순위'] = range(1, len(popularity_df) + 1)
            popularity_top50 = popularity_df.head(50)
            
            # 할당량 순위 (TOP 50)
            quota_df = pd.DataFrame(list(quotas.items()),
                                  columns=['메뉴명', '할당량'])
            quota_df = quota_df.sort_values('할당량', ascending=False)
            quota_df['순위'] = range(1, len(quota_df) + 1)
            quota_top50 = quota_df.head(50)
            
            # 카테고리별 통계
            category_stats = results_df.groupby('대분류').agg({
                '순수인기도': ['sum', 'mean', 'count'],
                '할당량': 'sum'
            }).round(1)
            
            # 인기도 구간별 분포
            def categorize_popularity(score):
                if score == 0:
                    return "0회 (언급없음)"
                elif score <= 5:
                    return "1-5회 (낮음)"
                elif score <= 20:
                    return "6-20회 (보통)"
                elif score <= 50:
                    return "21-50회 (높음)"
                else:
                    return "51회+ (매우높음)"
            
            results_df['인기도구간'] = results_df['순수인기도'].apply(categorize_popularity)
            popularity_distribution = results_df['인기도구간'].value_counts()
            
            with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
                # 전체 결과
                results_df.to_excel(writer, sheet_name='전체분석결과', index=False)
                
                # 인기도 TOP 50
                popularity_top50.to_excel(writer, sheet_name='인기도TOP50', index=False)
                
                # 할당량 TOP 50
                quota_top50.to_excel(writer, sheet_name='할당량TOP50', index=False)
                
                # 카테고리별 통계
                category_stats.to_excel(writer, sheet_name='카테고리별통계')
                
                # 인기도 분포
                popularity_distribution.to_frame('메뉴수').to_excel(writer, sheet_name='인기도분포')
                
                # 분석 요약
                months_str = f"{self.search_months[0]}-{self.search_months[-1]}월"
                zero_mention = len([s for s in popularity_scores.values() if s == 0])
                avg_popularity = sum(popularity_scores.values()) / len(popularity_scores)
                
                summary_data = {
                    '항목': [
                        '분석 기간',
                        '총 메뉴 수',
                        '언급된 메뉴 수',
                        '무언급 메뉴 수',
                        '총 언급 횟수',
                        '평균 언급 횟수',
                        '최고 인기 메뉴',
                        '목표 할당량 합계',
                        '기본 할당량',
                        '비례 할당량',
                        'API 요청 횟수',
                        '분석 완료 시각'
                    ],
                    '값': [
                        f"{self.search_year}년 {months_str}",
                        f"{len(results_df)}개",
                        f"{len(popularity_scores) - zero_mention}개",
                        f"{zero_mention}개",
                        f"{sum(popularity_scores.values())}회",
                        f"{avg_popularity:.1f}회",
                        f"{popularity_top50.iloc[0]['메뉴명']} ({popularity_top50.iloc[0]['순수인기도']}회)" if len(popularity_top50) > 0 else "없음",
                        f"{sum(quotas.values())}개",
                        "전체의 30% (최소 보장)",
                        "전체의 70% (인기도 비례)",
                        f"{self.daily_request_count}회",
                        datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    ]
                }
                
                summary_df = pd.DataFrame(summary_data)
                summary_df.to_excel(writer, sheet_name='분석요약', index=False)
                
                # 스타일 적용
                for sheet_name in writer.sheets:
                    worksheet = writer.sheets[sheet_name]
                    if worksheet.max_row > 1:
                        header_font = Font(bold=True)
                        for cell in worksheet[1]:
                            cell.font = header_font
            
            print(f"\n순수 인기도 분석 결과 저장 완료: {output_file}")
            print(f"총 {len(results_df)}개 메뉴 분석 결과 저장")
            
            # TOP 10 출력
            print(f"\n{self.search_year}년 {self.search_quarter}분기 순수 인기도 TOP 10:")
            print("-" * 60)
            for i, row in popularity_top50.head(10).iterrows():
                quota = quotas.get(row['메뉴명'], 0)
                print(f"{row['순위']:2d}. {row['메뉴명']:<15}: {row['순수인기도']:4d}회 → 할당량 {quota:3d}개")
            
            print(f"\n무언급 메뉴 수: {zero_mention}개")
            print(f"평균 언급 횟수: {avg_popularity:.1f}회")
            
        except Exception as e:
            print(f"결과 저장 실패: {e}")


def main():
    """메인 실행 함수"""
    CSV_FILE_PATH = "식당대12중53소132상세메뉴379분류.csv"
    SEARCH_YEAR = 2024
    SEARCH_QUARTER = 1  # 1분기 (1-3월)
    OUTPUT_FILE = f"menu_popularity_pure_{SEARCH_YEAR}Q{SEARCH_QUARTER}.xlsx"
    
    try:
        print("24년 1분기 메뉴별 순수 인기도 분석 시스템")
        print("="*50)
        
        # 분석기 초기화
        analyzer = MenuPopularityAnalyzer(
            search_year=SEARCH_YEAR,
            search_quarter=SEARCH_QUARTER
        )
        
        # 파일 확인
        if not os.path.exists(CSV_FILE_PATH):
            print(f"오류: CSV 파일을 찾을 수 없습니다 - {CSV_FILE_PATH}")
            return
        
        if not os.path.exists('.env'):
            print("오류: .env 파일이 필요합니다:")
            print("Client_ID=your_client_id")
            print("Client_Secret=your_client_secret")
            return
        
        # 순수 인기도 분석 실행
        popularity_scores, quotas = analyzer.analyze_all_menus(CSV_FILE_PATH)
        
        # 결과 저장
        analyzer.save_analysis_results(popularity_scores, quotas, OUTPUT_FILE)
        
        print("\n순수 인기도 분석 완료!")
        
    except Exception as e:
        print(f"오류 발생: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

24년 1분기 메뉴별 순수 인기도 분석 시스템
메뉴 인기도 분석기 초기화 완료
분석 기간: 2024년 1분기 (1-3월)
CSV 파일 로드 완료: 138개 행
총 381개의 개별 메뉴 항목 생성

2024년 1-3월 메뉴별 순수 인기도 분석 시작

[1/381] 제육볶음
  '제육볶음' 분석 중 (키워드 6개)
  총 언급: 0회

[2/381] 매운제육볶음
  '매운제육볶음' 분석 중 (키워드 6개)
    '매운제육볶음 맛집': 13회
    '매운제육볶음 추천': 5회
    '맛있는 매운제육볶음': 5회
  총 언급: 23회

[3/381] 두부제육볶음
  '두부제육볶음' 분석 중 (키워드 6개)
    '두부제육볶음 레시피': 18회
    '두부제육볶음 추천': 9회
    '맛있는 두부제육볶음': 10회
  총 언급: 37회

[4/381] 된장찌개
  '된장찌개' 분석 중 (키워드 6개)
  총 언급: 0회

[5/381] 김치찌개
  '김치찌개' 분석 중 (키워드 6개)
  총 언급: 0회

[6/381] 청국장찌개
  '청국장찌개' 분석 중 (키워드 6개)
    '청국장찌개 레시피': 5회
    '청국장찌개 추천': 6회
    '맛있는 청국장찌개': 6회
  총 언급: 17회

[7/381] 콩나물무침
  '콩나물무침' 분석 중 (키워드 6개)
    '콩나물무침 레시피': 8회
    '콩나물무침 추천': 14회
    '맛있는 콩나물무침': 22회
  총 언급: 44회

[8/381] 시금치나물
  '시금치나물' 분석 중 (키워드 6개)
    '시금치나물 맛집': 47회
    '시금치나물 레시피': 19회
    '맛있는 시금치나물': 68회
  총 언급: 134회

[9/381] 도라지무침
  '도라지무침' 분석 중 (키워드 6개)
    '도라지무침 맛집': 9회
    '도라지무침 요리': 28회
    '도라지무침 레시피': 9회
    '도라지무침 추천': 4회
    '맛있는 도라지무침': 26회
  총 언급: 76회


  총 언급: 0회

[100/381] 마른오징어
  '마른오징어' 분석 중 (키워드 6개)
    '마른오징어 맛집': 6회
    '마른오징어 레시피': 6회
    '마른오징어 추천': 23회
    '맛있는 마른오징어': 34회
  총 언급: 69회

진행률: 26.2% (100/381)
API 요청: 764회

[101/381] 육포
  '육포' 분석 중 (키워드 6개)
  총 언급: 0회

[102/381] 땅콩
  '땅콩' 분석 중 (키워드 6개)
  총 언급: 0회

[103/381] 신라면
  '신라면' 분석 중 (키워드 6개)
  총 언급: 0회

[104/381] 너구리
  '너구리' 분석 중 (키워드 6개)
  총 언급: 0회

[105/381] 짜파게티
  '짜파게티' 분석 중 (키워드 6개)
  총 언급: 0회

[106/381] 컵라면
  '컵라면' 분석 중 (키워드 6개)
  총 언급: 0회

[107/381] 용기면
  '용기면' 분석 중 (키워드 6개)
  총 언급: 0회

[108/381] 생라면
  '생라면' 분석 중 (키워드 6개)
    '생라면 맛집': 1회
    '생라면 요리': 2회
    '생라면 추천': 2회
  총 언급: 5회

[109/381] 쫄면
  '쫄면' 분석 중 (키워드 6개)
  총 언급: 0회

[110/381] 냉라면
  '냉라면' 분석 중 (키워드 6개)
    '냉라면': 7회
    '냉라면 맛집': 1회
    '냉라면 요리': 5회
    '냉라면 레시피': 6회
    '냉라면 추천': 1회
  총 언급: 20회

[111/381] 라멘
  '라멘' 분석 중 (키워드 6개)
  총 언급: 0회

[112/381] 우동
  '우동' 분석 중 (키워드 6개)
  총 언급: 0회

[113/381] 소바
  '소바' 분석 중 (키워드 6개)
  총 언급: 0회

[114/381] 짜장면
  '짜장면' 분석 중 (키워드 6개)
  총 언급: 0회

[115/381] 삼선짜장
  '삼선짜장'

    '맛있는 마제소바': 4회
  총 언급: 106회

[189/381] 매운미소
  '매운미소' 분석 중 (키워드 6개)
    '매운미소 추천': 39회
  총 언급: 39회

[190/381] 츠케멘
  '츠케멘' 분석 중 (키워드 6개)
    '츠케멘 맛집': 2회
    '츠케멘 요리': 2회
  총 언급: 4회

[191/381] 아부라소바
  '아부라소바' 분석 중 (키워드 6개)
    '아부라소바': 27회
    '아부라소바 맛집': 21회
    '아부라소바 요리': 6회
    '아부라소바 레시피': 5회
    '아부라소바 추천': 2회
    '맛있는 아부라소바': 1회
  총 언급: 62회

[192/381] 탄탄멘
  '탄탄멘' 분석 중 (키워드 6개)
    '탄탄멘 맛집': 4회
    '탄탄멘 레시피': 2회
    '탄탄멘 추천': 1회
    '맛있는 탄탄멘': 1회
  총 언급: 8회

[193/381] 가츠동
  '가츠동' 분석 중 (키워드 6개)
    '가츠동 맛집': 5회
    '가츠동 요리': 31회
    '가츠동 레시피': 2회
    '가츠동 추천': 3회
    '맛있는 가츠동': 2회
  총 언급: 43회

[194/381] 규동
  '규동' 분석 중 (키워드 6개)
    '규동 맛집': 44회
    '규동 요리': 35회
    '규동 레시피': 1회
    '규동 추천': 4회
    '맛있는 규동': 19회
  총 언급: 103회

[195/381] 오야코동
  '오야코동' 분석 중 (키워드 6개)
  총 언급: 0회

[196/381] 텐동
  '텐동' 분석 중 (키워드 6개)
    '텐동 맛집': 28회
    '텐동 추천': 3회
    '맛있는 텐동': 5회
  총 언급: 36회

[197/381] 가케우동
  '가케우동' 분석 중 (키워드 6개)
    '가케우동': 3회
    '가케우동 맛집': 3회
  총 언급: 6회

[198/381] 텐푸라우동
  '텐푸라우동' 분석 

    '레드커리 맛집': 5회
    '레드커리 레시피': 3회
    '레드커리 추천': 4회
    '맛있는 레드커리': 1회
  총 언급: 13회

진행률: 73.5% (280/381)
API 요청: 2261회

[281/381] 팬낭커리
  '팬낭커리' 분석 중 (키워드 6개)
  총 언급: 0회

[282/381] 똠얌꿍
  '똠얌꿍' 분석 중 (키워드 6개)
    '똠얌꿍 맛집': 14회
    '똠얌꿍 레시피': 3회
    '똠얌꿍 추천': 6회
    '맛있는 똠얌꿍': 3회
  총 언급: 26회

[283/381] 똠얌갈비
  '똠얌갈비' 분석 중 (키워드 6개)
    '똠얌갈비': 1회
    '똠얌갈비 맛집': 1회
    '똠얌갈비 요리': 1회
    '똠얌갈비 레시피': 1회
  총 언급: 4회

[284/381] 새콤매운국물
  '새콤매운국물' 분석 중 (키워드 6개)
    '새콤매운국물': 9회
    '새콤매운국물 맛집': 2회
    '새콤매운국물 요리': 4회
    '새콤매운국물 레시피': 2회
    '새콤매운국물 추천': 1회
    '맛있는 새콤매운국물': 1회
  총 언급: 19회

[285/381] 치킨커리
  '치킨커리' 분석 중 (키워드 6개)
  총 언급: 0회

[286/381] 양고기커리
  '양고기커리' 분석 중 (키워드 6개)
    '양고기커리 맛집': 8회
    '양고기커리 레시피': 21회
    '양고기커리 추천': 5회
    '맛있는 양고기커리': 5회
  총 언급: 39회

[287/381] 달커리
  '달커리' 분석 중 (키워드 6개)
    '달커리': 2회
    '달커리 요리': 1회
  총 언급: 3회

[288/381] 난
  '난' 분석 중 (키워드 6개)
  총 언급: 0회

[289/381] 파라타
  '파라타' 분석 중 (키워드 6개)
  총 언급: 0회

[290/381] 탄두리치킨
  '탄두리치킨' 분석 중 (키워드 6개)
    '탄두리치킨 맛집': 1회
 

    '퀴노아볼 추천': 2회
    '맛있는 퀴노아볼': 6회
  총 언급: 17회

[380/381] 프로틴볼
  '프로틴볼' 분석 중 (키워드 6개)
  총 언급: 0회

진행률: 99.7% (380/381)
API 요청: 3032회

[381/381] 그릭요거트
  '그릭요거트' 분석 중 (키워드 6개)
  총 언급: 0회

순수 인기도 분석 결과:
총 언급 횟수: 6043회
언급된 메뉴: 213개 / 381개

할당량 계산 방식:
기본 할당량: 6개 × 381메뉴 = 2286개
비례 할당량: 5714개 (인기도 비례 배분)
총 할당량: 7873개

분석 완료!
API 요청 총 3038회 사용

순수 인기도 분석 결과 저장 완료: menu_popularity_pure_2024Q1.xlsx
총 381개 메뉴 분석 결과 저장

2024년 1분기 순수 인기도 TOP 10:
------------------------------------------------------------
 1. 대게찜            :  189회 → 할당량 184개
 2. 시금치나물          :  134회 → 할당량 132개
 3. 삼선짬뽕           :  125회 → 할당량 124개
 4. 마라파스타          :  114회 → 할당량 113개
 5. 마제소바           :  106회 → 할당량 106개
 6. 규동             :  103회 → 할당량 103개
 7. 잡채밥            :  103회 → 할당량 103개
 8. 탄탄면            :   84회 → 할당량  85개
 9. 연어덮밥           :   82회 → 할당량  83개
10. 도라지무침          :   76회 → 할당량  77개

무언급 메뉴 수: 166개
평균 언급 횟수: 15.9회

순수 인기도 분석 완료!
